In [1]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
from factor_analyzer.rotator import Rotator

In [2]:
df = pd.read_csv("../data/wvs_time_series.csv", na_values=' ')
df.head()

,S025,A008,A165,E018,E025,F063,F118,F120,G006,Y002,Y003,survself,tradrat5,FAC1_1,FAC2_1
0,81998,3,2,1,2,10,1,5,2,2,-1,-1.011276,0.160847,-0.782357,0.217365
1,81998,2,2,2,3,8,2,5,2,2,0,-0.688773,0.409283,-0.521070,0.536164
2,81998,3,2,1,-1,9,1,5,2,2,-1,NaN,NaN,NaN,NaN
3,81998,3,2,1,2,9,1,4,3,2,-1,-1.319863,0.733956,-1.049071,0.821982
4,81998,3,1,1,-1,10,1,4,2,1,-2,NaN,NaN,NaN,NaN


In [3]:
df["tradagg"] = 1.61 * df["tradrat5"]  - .1
df["survsagg"] = 1.81 * df["survself"]  + .038
df.head()

,S025,A008,A165,E018,E025,F063,F118,F120,G006,Y002,Y003,survself,tradrat5,FAC1_1,FAC2_1,tradagg,survsagg
0,81998,3,2,1,2,10,1,5,2,2,-1,-1.011276,0.160847,-0.782357,0.217365,0.158964,-1.792410
1,81998,2,2,2,3,8,2,5,2,2,0,-0.688773,0.409283,-0.521070,0.536164,0.558946,-1.208679
2,81998,3,2,1,-1,9,1,5,2,2,-1,NaN,NaN,NaN,NaN,NaN,NaN
3,81998,3,2,1,2,9,1,4,3,2,-1,-1.319863,0.733956,-1.049071,0.821982,1.081669,-2.350952
4,81998,3,1,1,-1,10,1,4,2,1,-2,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ok at least this works
df.groupby("S025")[["tradagg", "survsagg"]].mean().head()

,tradagg,survsagg
S025,,
81998,0.576952,-1.009137
82002,0.138196,-0.785331
122002,-1.263754,-1.202245
122014,-0.775223,-0.927439
202005,0.782631,1.865386


In [5]:
for col in ["A008", "A165", "E018", "E025", "F063", "F118", "F120", "G006", "Y002"]:
    df[col] = df[col].replace(list(range(-9, 0)), np.nan)

df["Y003"] = df["Y003"].replace(-5, np.nan)

In [6]:
vars_for_pca = ["A008", "A165", "E018", "E025", "F063", "F118", "F120", "G006", "Y002", "Y003"]

In [7]:
df_complete = df.dropna(subset=vars_for_pca).copy()

Xz = StandardScaler().fit_transform(df_complete[vars_for_pca])

pca = PCA(n_components=2)
scores = pca.fit_transform(Xz)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

rotator = Rotator(method="varimax", normalize=True)
rot_loadings = rotator.fit_transform(loadings)

# Approximate rotated scores
R = np.linalg.lstsq(loadings, rot_loadings, rcond=None)[0]
rot_scores = scores @ R

In [8]:
rot_scores[:5]

array([[-1.12313796,  0.21840399],
       [-0.64183427,  0.62568398],
       [-1.31261685,  0.97872724],
       [-1.61833188,  0.87711719],
       [-0.32937072,  0.45673298]])